# 02 · Final model and submission file (cached features only)

Loads the features built by `01_extract_features` (Whisper is **not** run here), trains small LightGBM models, checks them with repeated stratified 5-fold CV, and writes a validated `submission.csv`.

**Why there are variants.** Inspection of the training data showed two things that matter for the test set:
1. All 37 clips labelled 0 come from a separate batch of file names (ID >= 5000); no test file belongs to that batch, and the constant-guess leaderboard score (about 1.0) fits a test set without them.
2. Test clips are shorter than training clips (median 45 s vs 60 s), so features that grow with clip length (word counts, pause counts, duration) are shifted.

| VARIANT | features | training clips |
|---|---|---|
| `A` | all features | all 769 |
| `B` | length-dependent features removed | label-0 batch removed (732) |
| `C` | length-dependent features removed | all 769 |

Change `VARIANT` in the next cell, run all, and submit. Public leaderboard scores of the variants are then compared.

In [ ]:
# ===================== SETTINGS: the only things to edit =====================
VARIANT   = "B"            # "A", "B" or "C" (see the table above)
DATA_DIR  = "/kaggle/input/competitions/shl-hiring-assessment-2026/Dataset_Final"
WORK_DIR  = "/kaggle/working"
N_SEEDS   = 5              # LightGBM seeds averaged in the final model
CV_SEEDS  = [42, 43, 44]   # repeated CV

import os, sys, shutil, warnings
from pathlib import Path
warnings.filterwarnings("ignore", message="X does not have valid feature names")

def _find_repo():
    cands = [Path.cwd(), Path(WORK_DIR) / "shl-grammar-scoring"]
    for pat in ("*", "*/*", "*/*/*", "*/*/*/*"):               # Kaggle nests datasets 1-4 levels deep
        cands += list(Path("/kaggle/input").glob(pat))
    for c in cands:
        if (c / "src" / "config.py").exists():
            return c
    raise FileNotFoundError("Cannot find the repo's src/ folder: attach the code dataset (Add Input).")

REPO_DIR = _find_repo(); sys.path.insert(0, str(REPO_DIR))

CACHE = Path(WORK_DIR) / "cache"
if not (CACHE / "features" / "meta.csv").exists():             # copy the features made by 01 (attached as an input)
    hits = list(Path("/kaggle/input").rglob("features/meta.csv"))
    assert hits, "Features not found: attach the COMMITTED 01_extract_features notebook (Add Input -> Your Work)."
    shutil.copytree(hits[0].parent, CACHE / "features", dirs_exist_ok=True)

from src.config import Config
cfg = Config(DATA_DIR=DATA_DIR, CACHE_DIR=str(CACHE), OUTPUT_DIR=WORK_DIR).apply_env().seed_everything()
print("repo    :", REPO_DIR)
print("features:", cfg.features_dir, "| variant:", VARIANT)

## 1 · Load the cached features and choose the variant

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import display
from src.data import discover_layout
from src.models import run_cv, make_lgbm, stratify_labels
from src.metrics import rmse, pearson, clip_scores
from src.submission import write_submission

fd = cfg.features_dir
meta = pd.read_csv(fd / "meta.csv")
F = pd.read_csv(fd / "features.csv")
assert len(meta) == len(F), "features and meta are misaligned: re-run 01"
tr = (meta.split == "train").to_numpy(); te = ~tr
y_all = meta.loc[tr, "label"].to_numpy(float)

# features whose size grows with clip length (test clips are shorter than training clips)
LENGTH_COLS = [c for c in ("flu_audio_duration", "flu_speech_span", "flu_n_segments", "flu_pause_count",
                           "flu_long_pause_count", "hand_n_words", "hand_n_sentences", "hand_unique_ratio")
               if c in F.columns]
RATE_COLS = [c for c in F.columns if c not in LENGTH_COLS]

if VARIANT == "A":   cols, keep = list(F.columns), np.ones(tr.sum(), bool)
elif VARIANT == "B": cols, keep = RATE_COLS, y_all > 0
elif VARIANT == "C": cols, keep = RATE_COLS, np.ones(tr.sum(), bool)
else: raise ValueError("VARIANT must be 'A', 'B' or 'C'")

Xtr = F.loc[tr, cols].to_numpy(float); Xte = F.loc[te, cols].to_numpy(float)
Xk, yk = Xtr[keep], y_all[keep]
print(f"variant {VARIANT}: {len(cols)} features | {int(keep.sum())} of {len(keep)} training clips used")
print("length-dependent columns removed:" if VARIANT != "A" else "length-dependent columns (kept):", LENGTH_COLS)
print(f"constant-baseline RMSE (label std of the clips used): {yk.std():.4f}")

## 2 · Cross-validation (repeated stratified 5-fold, seeds fixed)

In [ ]:
strat = stratify_labels(yk, cfg.CV_SPLITS)
cv = run_cv(make_lgbm, Xk, yk, strat, CV_SEEDS, cfg.CV_SPLITS, f"lgbm_variant_{VARIANT}")
s = cv["summary"]

nz = yk > 0                                   # test-like view: score only the clips with label > 0
rmse_nz = float(np.mean([rmse(yk[nz], o[nz]) for o in cv["oof_repeats"]]))
pear_nz = float(np.nanmean([pearson(yk[nz], o[nz]) for o in cv["oof_repeats"]]))
print(f"CV on the clips used     : RMSE {s['rmse_overall']:.4f} | Pearson {s['pearson_overall']:.4f}")
print(f"CV on label>0 clips only : RMSE {rmse_nz:.4f} | Pearson {pear_nz:.4f}   <- closest to the test set")
print(f"fold-to-fold std of RMSE : {s['rmse_fold_std']:.4f}  (differences smaller than this are noise)")

## 3 · Final fit on all selected training clips, training RMSE, test prediction

In [ ]:
models = [make_lgbm(42 + k).fit(Xk, yk) for k in range(N_SEEDS)]
predict = lambda X: clip_scores(np.mean([m.predict(X) for m in models], axis=0))

train_pred_used = predict(Xk)
train_pred_all  = predict(Xtr)
test_pred = predict(Xte)

print("=" * 72)
print(f"TRAINING RMSE (clips used for training)       : {rmse(yk, train_pred_used):.4f}")
print(f"TRAINING RMSE (all {len(y_all)} training clips)       : {rmse(y_all, train_pred_all):.4f}")
print(f"CV RMSE / CV Pearson (clips used)             : {s['rmse_overall']:.4f} / {s['pearson_overall']:.4f}")
print(f"CV RMSE / CV Pearson (label>0 clips only)     : {rmse_nz:.4f} / {pear_nz:.4f}")
print(f"Constant baseline RMSE (label std, clips used): {yk.std():.4f}")
print("=" * 72)
print("(training RMSE is optimistic: the model has seen these labels. CV RMSE is the honest number.)")

## 4 · Write and validate `submission.csv`
One row per `test.csv` row, in that order (the leaderboard rejects the 204-row `sample_submission.csv`).

In [ ]:
template = discover_layout(cfg.DATA_DIR)["test_csv"]
sub = write_submission(test_pred, meta[te].reset_index(drop=True), template, str(Path(WORK_DIR) / "submission.csv"))
display(sub.head())
print(sub.iloc[:, -1].describe().round(3).to_string())
print(f"mean prediction on test {test_pred.mean():.3f} vs mean training label used {yk.mean():.3f}")

## 5 · Plots

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16, 4.2))
oof = cv["oof"]
ax[0].scatter(yk, oof, s=10, alpha=.5); ax[0].plot([0, 5], [0, 5], "k--", lw=1)
ax[0].set(xlabel="true score", ylabel="out-of-fold prediction", title=f"Variant {VARIANT}: out-of-fold predictions")
ax[1].hist(oof - yk, bins=30); ax[1].set(xlabel="prediction - truth", title="Residuals (out-of-fold)")
ax[2].hist(yk, bins=np.arange(-.25, 5.5, .5), alpha=.6, density=True, label="training labels")
ax[2].hist(test_pred, bins=np.arange(-.25, 5.5, .5), alpha=.6, density=True, label="test predictions"); ax[2].legend()
ax[2].set(xlabel="score", title="Label vs prediction distribution")
plt.tight_layout(); plt.savefig(Path(WORK_DIR) / f"variant_{VARIANT}_diagnostics.png", dpi=120); plt.show()

try:
    imp = pd.Series(np.mean([m.feature_importances_ for m in models], axis=0), index=cols).sort_values()
    fig, ax = plt.subplots(figsize=(7, 7)); imp.tail(20).plot.barh(ax=ax)
    ax.set(title="Feature importance (LightGBM splits, top 20)"); plt.tight_layout()
    plt.savefig(Path(WORK_DIR) / f"variant_{VARIANT}_importance.png", dpi=120); plt.show()
except Exception as e:
    print("feature importance not available for this model:", e)

## 6 · Short report

* **Pipeline:** Whisper transcripts and timestamps (cached by `01`) -> engineered text / fluency / ASR-confidence / perplexity / grammar-checker features -> LightGBM (shallow, regularised, averaged over several seeds) -> predictions clipped to [0, 5].
* **Validation:** repeated stratified 5-fold CV; all numbers printed above come from this run.
* **Variant choice:** training clips labelled 0 form a separate batch (file ID >= 5000) that the test files do not appear to contain, and test clips are shorter than training clips. Variant `B` removes both risks; the public leaderboard score is the final check.
* **Limits:** only about 770 labelled clips; fold-to-fold RMSE varies by several hundredths, so tiny CV differences should not be over-interpreted; the test set cannot be inspected for label-0-like clips.